# RoBERTa Fine-tuning
## Domain-Adapted Fake News Detection

Fine-tune pretrained RoBERTa-base trên dữ liệu tin tức để phát hiện tin giả.

> **[WARNING] GPU Required**: Notebook này cần GPU. Nếu chạy trên Colab, chọn Runtime → T4 GPU.
> Upload thư mục `src/` và `data/processed/` lên Colab trước khi chạy.


In [ ]:
# ============================================================
# COLAB SETUP - Chay cell nay TRUOC TIEN neu dung Colab
# ============================================================
import os
import sys
import zipfile

# Check if running on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print('[OK] Running on Google Colab')
except:
    IN_COLAB = False
    print('Running locally (not Colab)')

if IN_COLAB:
    # Mount Google Drive
    print('\nMounting Google Drive...')
    drive.mount('/content/drive')
    
    ZIP_PATH = '/content/drive/MyDrive/Fake_news_RoBERTa.zip'
    EXTRACT_DIR = '/content/project'
    
    if os.path.exists(ZIP_PATH):
        print(f'\n[OK] Found zip file: {ZIP_PATH}')
        print(f'  Size: {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB')
        
        # Clean old extraction
        if os.path.exists(EXTRACT_DIR):
            import shutil
            shutil.rmtree(EXTRACT_DIR)
        
        # Extract using Python zipfile (handles Windows backslash paths)
        print('\nExtracting files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            for info in zf.infolist():
                info.filename = info.filename.replace('\\', '/')
                zf.extract(info, EXTRACT_DIR)
        print('[OK] Extraction complete.')
        
        # Setup paths
        PROJECT_ROOT = EXTRACT_DIR
        sys.path.insert(0, PROJECT_ROOT)
        
        # Verify
        print(f'\n[OK] Project root: {PROJECT_ROOT}')
        checks = {
            'src/config.py': True,
            'data/processed/train.csv': True,
            'data/processed/val.csv': True,
            'data/processed/test.csv': True,
        }
        for f, required in checks.items():
            exists = os.path.exists(os.path.join(PROJECT_ROOT, f))
            status = '[OK]' if exists else ('[MISSING]' if required else '[NOT YET]')
            print(f'  {status} {f}')
        print('\n[OK] Ready for training!')
    else:
        print(f'\n[ERROR] Zip not found: {ZIP_PATH}')
        print('Upload Fake_news_RoBERTa.zip to Google Drive root.')
else:
    print('\nSkipping Colab setup (running locally)')
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)


In [ ]:
# Imports
import os
import sys

# Ensure project root is in path
try:
    _ = PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)

import torch
from torch.utils.data import DataLoader

from src.config import *
from src.utils import set_seed, get_device, print_gpu_memory
from src.dataset import prepare_datasets
from src.model import load_model_and_tokenizer, save_model, get_model_summary, freeze_layers
from src.trainer import Trainer
from src.evaluation import (
    evaluate_model, print_evaluation_report,
    plot_confusion_matrix, plot_training_history, plot_roc_curve
)

set_seed(SEED)
device = get_device()


## 1. Load Model & Tokenizer


In [ ]:
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout_rate=DROPOUT_RATE
)
get_model_summary(model)


## 2. Prepare Datasets


In [ ]:
train_dataset, val_dataset, test_dataset = prepare_datasets(
    train_file=TRAIN_FILE,
    val_file=VAL_FILE,
    test_file=TEST_FILE,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nBatch size: {BATCH_SIZE}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')


## 3. (Optional) Freeze Bottom Layers

Đóng băng các layers đầu để giữ kiến thức tổng quát, chỉ fine-tune layers cuối.


In [ ]:
# Uncomment to freeze bottom 6 layers (of 12)
# model = freeze_layers(model, num_layers_to_freeze=6)
# get_model_summary(model)


## 4. Training


In [ ]:
trainer = Trainer(
    model=model,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    device=device,
    learning_rate=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)

history = trainer.train()


## 5. Training History Visualization


In [ ]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
plot_training_history(history, save_path=os.path.join(RESULTS_DIR, 'roberta_training_history.png'))


## 6. Evaluate on Test Set


In [ ]:
predictions, true_labels, all_probs, metrics = evaluate_model(model, test_loader, device)

print_evaluation_report(predictions, true_labels)

# Save metrics
from src.utils import save_metrics
roberta_results = {
    'model': 'RoBERTa (fine-tuned)',
    **{k: float(v) for k, v in metrics.items()}
}
save_metrics(roberta_results, os.path.join(RESULTS_DIR, 'roberta_results.json'))


In [ ]:
# Confusion Matrix
plot_confusion_matrix(predictions, true_labels,
    save_path=os.path.join(RESULTS_DIR, 'roberta_confusion_matrix.png'),
    title='RoBERTa — Confusion Matrix')


In [ ]:
# ROC Curve
roc_auc = plot_roc_curve(true_labels, all_probs,
    save_path=os.path.join(RESULTS_DIR, 'roberta_roc_curve.png'),
    title='RoBERTa — ROC Curve')
print(f'ROC AUC: {roc_auc:.4f}')


## 7. Save Trained Model


In [ ]:
save_model(model, tokenizer, ROBERTA_MODEL_DIR)
print(f'\n[OK] Model saved to: {ROBERTA_MODEL_DIR}')


## 8. Quick Sanity Check


In [ ]:
from src.evaluation import predict_single

test_texts = [
    "The president announced new economic policies today during the press conference.",
    "BREAKING: Scientists discover that the moon is actually made of cheese!",
]

for text in test_texts:
    result = predict_single(model, tokenizer, text, device)
    print(f'\nText: {text[:80]}...')
    print(f'  Prediction: {result["prediction"]} ({result["confidence"]:.1%})')
    print(f'  Probs: real={result["probabilities"]["real"]:.3f}, fake={result["probabilities"]["fake"]:.3f}')

print('\n[OK] RoBERTa fine-tuning complete!')
